In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for _ in range(5):
    if (PROJECT_ROOT / "pyproject.toml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists(), "Could not find project root"
sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.getLogger("src").setLevel(logging.WARNING)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["figure.dpi"] = 100

## Architecture Overview

**TranAD** (Transformer-based Anomaly Detection) uses a shared transformer encoder paired with two separate decoders for **two-phase self-conditioning**:

1. **Phase 1** -- The encoder processes the input window concatenated with a zero "focus score." Decoder 1 produces reconstruction `x1`. Because the focus is all zeros, this is a standard reconstruction.

2. **Phase 2** -- The squared error `(x1 - input)^2` from Phase 1 becomes the focus score. The encoder re-processes the window concatenated with this error signal, and Decoder 2 produces a refined reconstruction `x2`. This forces Decoder 2 to attend more strongly to the regions where Phase 1 struggled.

This two-phase design creates **sharper anomaly separation**: normal points are reconstructed well in both phases, while anomalous points accumulate compounding error. The training loss evolves over epochs -- early training weights Phase 1 heavily (`w = 1/epoch`), and later training shifts weight to Phase 2, producing increasingly sharp anomaly scores.

```
Input window (W, B, F)
        |
        v
  [concat with zeros] --> Encoder --> Decoder 1 --> x1 (Phase 1)
        |                                |
        |                          (x1 - input)^2
        |                                |
  [concat with error] --> Encoder --> Decoder 2 --> x2 (Phase 2)
        |                                |
        v                                v
  Shared projection: Linear(2F, F) + Sigmoid
```

## Instantiate Model

In [2]:
from src.model import TranADConfig, TranADNet

config = TranADConfig()  # defaults: 38 features, window_size=10
model = TranADNet(config)

print("=== TranAD Architecture ===")
print(model)
print()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nConfig: window_size={config.window_size}, n_features={config.n_features}, "
      f"d_model={config.d_model}, n_heads={config.n_heads}, d_ff={config.d_feedforward}")

=== TranAD Architecture ===
TranADNet(
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=76, out_features=76, bias=True)
        )
        (linear1): Linear(in_features=76, out_features=16, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=16, out_features=76, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (activation): LeakyReLU(negative_slope=True)
      )
    )
  )
  (transformer_decoder1): TransformerDecoder(
    (layers): ModuleList(
      (0): TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=76, out_features=76, bias=True)
        )
  

## Forward Pass Demo

The model expects:
- `src`: sliding window of shape `(window_size, batch, n_features)`
- `tgt`: the last element of the window, shape `(1, batch, n_features)`

It returns two outputs: `x1` (Phase 1 reconstruction) and `x2` (Phase 2 reconstruction), both of shape `(1, batch, n_features)`.

In [3]:
# Create random input matching expected shapes
window_size, batch_size, n_features = 10, 4, 38

src = torch.randn(window_size, batch_size, n_features)
tgt = src[-1:, :, :]  # last timestep as target

print(f"Input  src shape: {tuple(src.shape)}  (window_size, batch, features)")
print(f"Input  tgt shape: {tuple(tgt.shape)}  (1, batch, features)")

model.eval()
with torch.no_grad():
    x1, x2 = model(src, tgt)

print(f"\nOutput x1 shape:  {tuple(x1.shape)}  (Phase 1 reconstruction)")
print(f"Output x2 shape:  {tuple(x2.shape)}  (Phase 2 reconstruction)")
print(f"\nx1 = Phase 1: standard reconstruction (focus = zeros)")
print(f"x2 = Phase 2: refined reconstruction (focus = Phase 1 squared error)")
print(f"\nOutput range: x1 in [{x1.min():.4f}, {x1.max():.4f}], "
      f"x2 in [{x2.min():.4f}, {x2.max():.4f}]")
print(f"(Both pass through Sigmoid, so values are in [0, 1])")

Input  src shape: (10, 4, 38)  (window_size, batch, features)
Input  tgt shape: (1, 4, 38)  (1, batch, features)

Output x1 shape:  (1, 4, 38)  (Phase 1 reconstruction)
Output x2 shape:  (1, 4, 38)  (Phase 2 reconstruction)

x1 = Phase 1: standard reconstruction (focus = zeros)
x2 = Phase 2: refined reconstruction (focus = Phase 1 squared error)

Output range: x1 in [0.0494, 0.9168], x2 in [0.0015, 0.9998]
(Both pass through Sigmoid, so values are in [0, 1])


## Load Real Data

Load preprocessed machine-1-1 data and build sliding windows for training and inference.

In [4]:
from src.preprocess import load_processed_data
from src.utils import convert_to_windows

MACHINE = "machine-1-1"
DATA_DIR = PROJECT_ROOT / "data" / "smd" / "processed"

train_data, test_data, test_labels, interp_labels = load_processed_data(DATA_DIR, MACHINE)

print(f"Machine: {MACHINE}")
print(f"  Train: {train_data.shape} ({train_data.shape[0]:,} timesteps x {train_data.shape[1]} features)")
print(f"  Test:  {test_data.shape}")
print(f"  Test labels: {test_labels.shape}")
print(f"  Anomaly rate: {test_labels.mean():.2%}")

# Build sliding windows for training
train_tensor = torch.from_numpy(train_data).float()
train_windows = convert_to_windows(train_tensor, config.window_size)
print(f"\nSliding windows: {tuple(train_windows.shape)}  (N, window_size, features)")

Machine: machine-1-1
  Train: (28479, 38) (28,479 timesteps x 38 features)
  Test:  (28479, 38)
  Test labels: (28479,)
  Anomaly rate: 9.46%

Sliding windows: (28479, 10, 38)  (N, window_size, features)


## Quick Training Demo

Train for 3 epochs on machine-1-1 to demonstrate the training loop. In practice, 5 epochs is the default (converges quickly for this architecture).

In [5]:
from torch.utils.data import DataLoader, TensorDataset
from src.train import train_epoch

# Fresh model for training demo
demo_config = TranADConfig(epochs=3)
demo_model = TranADNet(demo_config)

device = torch.device("cpu")
demo_model = demo_model.to(device)

optimizer = torch.optim.AdamW(
    demo_model.parameters(),
    lr=demo_config.lr,
    weight_decay=demo_config.weight_decay,
)
loss_fn = nn.MSELoss(reduction="none")

# DataLoader
dataset = TensorDataset(train_windows)
dataloader = DataLoader(dataset, batch_size=demo_config.batch_size, shuffle=True)

print(f"Training for {demo_config.epochs} epochs on {MACHINE} ({len(dataset):,} windows)...")
print(f"Batch size: {demo_config.batch_size}, Batches per epoch: {len(dataloader)}")
print()

losses = []
for epoch in range(demo_config.epochs):
    avg_loss = train_epoch(
        demo_model, dataloader, optimizer, loss_fn,
        epoch=epoch, config=demo_config, device=device,
    )
    losses.append(avg_loss)
    print(f"  Epoch {epoch + 1}/{demo_config.epochs}: loss = {avg_loss:.6f}")

print(f"\nLoss decreased: {losses[0]:.6f} -> {losses[-1]:.6f} "
      f"({(1 - losses[-1]/losses[0]) * 100:.1f}% reduction)")

Training for 3 epochs on machine-1-1 (28,479 windows)...
Batch size: 128, Batches per epoch: 223

  Epoch 1/3: loss = 0.036063
  Epoch 2/3: loss = 0.013727
  Epoch 3/3: loss = 0.005126

Loss decreased: 0.036063 -> 0.005126 (85.8% reduction)


## Score and Visualize

Load the pre-trained model (trained for 5 epochs) and score the test set. The anomaly score is the per-timestep mean reconstruction error across all 38 features.

In [ ]:
from src.registry import TranADRegistry
from src.scorer import score_batch

# Load pre-trained model
registry = TranADRegistry(base_dir=PROJECT_ROOT / "models" / "tranad")
pretrained_model, pretrained_config = registry.get_model(MACHINE, device="cpu")
print(f"Loaded pre-trained model for {MACHINE}")

# Score test data -> per-dimension MSE, shape (N, 38)
test_scores = score_batch(
    pretrained_model, test_data,
    window_size=pretrained_config.window_size,
    device="cpu",
    scoring_mode=pretrained_config.scoring_mode,
)
print(f"Test scores shape: {test_scores.shape}")

# Aggregate to 1D score
score_1d = np.mean(test_scores, axis=1)
print(f"Aggregated score range: [{score_1d.min():.6f}, {score_1d.max():.6f}]")

# Plot
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(score_1d, linewidth=0.5, color="steelblue", label="Anomaly score")

# Overlay ground truth anomaly regions
anomaly_mask = test_labels > 0 if test_labels.ndim == 1 else test_labels.sum(axis=1) > 0
ax.fill_between(
    range(len(score_1d)), 0, score_1d.max(),
    where=anomaly_mask, alpha=0.15, color="red", label="Ground truth anomaly",
)

ax.set_xlabel("Timestep")
ax.set_ylabel("Anomaly Score (mean MSE)")
ax.set_title(f"{MACHINE} -- Anomaly Scores (pre-trained model)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Threshold Calibration

**POT (Peaks-Over-Threshold)** is an extreme value theory method used to set the anomaly threshold:

1. Fit a Generalized Pareto Distribution (GPD) to the tail of the training score distribution
2. Use the fitted GPD to estimate a threshold at a desired quantile level
3. Apply a scaling factor for safety margin

POT is preferred over fixed percentiles because it models the tail behavior specifically -- it adapts to the shape of the score distribution rather than assuming a fixed fraction of anomalies.

The calibrated threshold and parameters are stored in `scorer_state.json`.

In [ ]:
import json

# Load calibrated threshold from scorer_state.json
scorer_state = registry.get_scorer_state(MACHINE)
threshold = scorer_state["threshold"]
pot_details = scorer_state["details"]

print(f"Threshold method: {scorer_state['method']}")
print(f"Calibrated threshold: {threshold:.6f}")
print(f"POT parameters: q={pot_details['q']}, level={pot_details['level']}, scale={pot_details['scale']}")

# Re-draw score plot with threshold line
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(score_1d, linewidth=0.5, color="steelblue", label="Anomaly score")

# Ground truth regions
ax.fill_between(
    range(len(score_1d)), 0, score_1d.max(),
    where=anomaly_mask, alpha=0.15, color="red", label="Ground truth anomaly",
)

# Threshold line
ax.axhline(y=threshold, color="darkred", linestyle="--", linewidth=1.5,
           label=f"POT threshold = {threshold:.4f}")

ax.set_xlabel("Timestep")
ax.set_ylabel("Anomaly Score (mean MSE)")
ax.set_title(f"{MACHINE} -- Anomaly Scores with POT Threshold")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Statistics
predictions = (score_1d > threshold).astype(int)
print(f"\nTimesteps above threshold: {predictions.sum():,} / {len(predictions):,} ({predictions.mean():.2%})")

## Attribution Demo

For a detected anomaly segment, we identify **root cause features** using the **elevation ratio**:

$$\text{elevation}_i = \frac{\text{score}_i}{\text{baseline}_i}$$

where `baseline_i` is the 95th percentile of feature `i`'s reconstruction error on training data. Features with high elevation ratios are reconstructed much worse than normal -- these are the likely root causes.

This normalization prevents features that are inherently hard to reconstruct (high baseline error) from always dominating the ranking.

## Evaluation Metrics

With the threshold calibrated, we can evaluate detection performance using the **point-adjustment protocol** -- the standard evaluation method for time series anomaly detection. If *any* point within a ground truth anomaly segment is correctly predicted, *all* points in that segment are credited as detected.

We report results across all 4 reference SMD machines.

In [ ]:
import json

# Load saved evaluation results for all 4 reference machines
MACHINES = ["machine-1-1", "machine-2-1", "machine-3-2", "machine-3-7"]
model_dir = PROJECT_ROOT / "models" / "tranad"

results = {}
for machine in MACHINES:
    eval_path = model_dir / machine / "eval_results.json"
    if eval_path.exists():
        with open(eval_path) as f:
            results[machine] = json.load(f)

# Print results table
print(f"{'Machine':<16} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC AUC':>9}")
print("-" * 55)

f1s, precs, recs = [], [], []
for machine in MACHINES:
    if machine in results:
        m = results[machine]
        print(f"{machine:<16} {m['precision']:>10.4f} {m['recall']:>8.4f} "
              f"{m['f1']:>8.4f} {m['roc_auc']:>9.4f}")
        f1s.append(m["f1"])
        precs.append(m["precision"])
        recs.append(m["recall"])

print("-" * 55)
print(f"{'Average':<16} {np.mean(precs):>10.4f} {np.mean(recs):>8.4f} {np.mean(f1s):>8.4f}")
print(f"\nAverage F1: {np.mean(f1s):.3f}")

In [ ]:
from src.scorer import find_anomaly_segments, attribute_dimensions

# Load baselines from scorer_state
baselines = np.array(scorer_state["feature_baselines"])
print(f"Feature baselines shape: {baselines.shape}")
print(f"Baseline range: [{baselines.min():.6f}, {baselines.max():.6f}]")

# Find anomaly segments from predictions
segments = find_anomaly_segments(predictions)
print(f"\nDetected anomaly segments: {len(segments)}")

# Pick the segment with the highest peak score for demonstration
if segments:
    peak_scores = [score_1d[s:e+1].max() for s, e in segments]
    best_seg_idx = np.argmax(peak_scores)
    seg_start, seg_end = segments[best_seg_idx]
    print(f"Analyzing segment {best_seg_idx + 1}: timesteps [{seg_start}, {seg_end}] "
          f"(length={seg_end - seg_start + 1}, peak={peak_scores[best_seg_idx]:.6f})")

    # Extract per-dimension scores for this segment
    segment_scores = test_scores[seg_start:seg_end + 1]  # (T, 38)

    # Compute elevation ratios
    elevation_ratios = segment_scores / baselines  # (T, 38)
    mean_elevation = np.mean(elevation_ratios, axis=0)  # (38,)

    # Get top 5 features by elevation
    top_indices = np.argsort(mean_elevation)[::-1][:5]

    print(f"\nTop 5 attributed features (by mean elevation ratio):")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank}. dim_{idx}: elevation = {mean_elevation[idx]:.2f}x "
              f"(score={np.mean(segment_scores[:, idx]):.6f}, baseline={baselines[idx]:.6f})")

    # Bar chart of top 5
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = [f"dim_{idx}" for idx in top_indices]
    values = [mean_elevation[idx] for idx in top_indices]
    colors = ["#d62728" if v > 5 else "#ff7f0e" if v > 2 else "#2ca02c" for v in values]

    bars = ax.bar(labels, values, color=colors, edgecolor="black", linewidth=0.5)
    ax.axhline(y=1.0, color="gray", linestyle="--", linewidth=1, label="Baseline (1.0x)")
    ax.axhline(y=2.0, color="orange", linestyle=":", linewidth=1, label="Min elevation threshold (2.0x)")

    ax.set_xlabel("Feature")
    ax.set_ylabel("Mean Elevation Ratio (score / baseline)")
    ax.set_title(f"Root Cause Attribution -- Segment [{seg_start}, {seg_end}]")
    ax.legend()

    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{val:.1f}x", ha="center", va="bottom", fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.show()

## Summary

TranAD's two-phase architecture produces per-dimension reconstruction errors that enable both **detection** (aggregated score vs threshold) and **diagnosis** (elevation-ratio ranking). The full pipeline:

**raw data** --> **normalize** --> **sliding windows** --> **TranAD inference** --> **per-dim MSE** --> **aggregate** --> **POT threshold** --> **segment detection** --> **elevation attribution**